In [1]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import cv2
import re
import pandas as pd

from functions import *

In [2]:
# Input
folder_in_main = '/media/joris/rootfs/home/joris/data/kiwibes2026/20260706kiwibes/bes21-30-joris'

In [3]:
# Prepare machine and segmentation model
predictor = prepare_model(verbose = False)

In [ ]:
results = []

# Loop over berries
for folder_in_berry in glob.glob(f'{folder_in_main}/*'):
    
    # Process all images of one berry
    berry_name = folder_in_berry.split('/')[-1]
    print()
    print(berry_name)
    
    if not os.path.exists(berry_name):
        os.mkdir(berry_name)

    ellipses_berry = []
    
    for file_in in glob.glob(f'{folder_in_berry}/*'):
    
        # Process one image
        print('  ', re.sub(folder_in_main, '', file_in))
        img = cv2.imread(file_in)
        grid_spacing = find_grid_spacing(img, visualize = False)
        mask = make_mask(img, predictor, visualize = False)

        file_plot = f'{berry_name}/{file_in.split("/")[-1]}'
        
        if not mask is None:
            ellipse = fit_ellipse(mask)
            # print(ellipse)
            ellipses_berry.append(ellipse)
        
            plt.figure()
            plt.subplot(131)
            plt.imshow(img)
            plt.axis('off')
            plt.subplot(132)
            plt.imshow(mask)
            plt.axis('off')
            plt.subplot(133)
            plot_mask_ellipse(mask, ellipse)
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(file_plot)
            plt.close()
            # plt.show()
    
        else:
            plt.figure()
            plt.subplot(131)
            plt.imshow(img)
            plt.axis('off')
            plt.subplot(132)
            plt.imshow(np.zeros(img.shape))
            plt.axis('off')
            plt.subplot(133)
            plt.imshow(np.zeros(img.shape))
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(file_plot)
            plt.close()
            # plt.show()
    
    # Select three main radii (mm)
    r1 = np.max([ellipse['semi_major'] for ellipse in ellipses_berry])*50/grid_spacing
    r2 = np.max([ellipse['semi_minor'] for ellipse in ellipses_berry])*50/grid_spacing
    r3 = np.min([ellipse['semi_minor'] for ellipse in ellipses_berry])*50/grid_spacing
    
    # Calculate volume (mm**3)
    vol = 4*np.pi/3*r1*r2*r3
    
    # Append result
    result = {
        'berry_name': berry_name,
        'grid_spacing': grid_spacing,
        'r1 (mm)': r1,
        'r2 (mm)': r2,
        'r3 (mm)': r3,
        'volume (mm**3)': vol
    }
    results.append(result)



bes29
   /bes29/20260709_143701.jpg
   /bes29/20260709_143640.jpg
   /bes29/20260709_143631.jpg
   /bes29/20260709_143639.jpg
   /bes29/20260709_143705.jpg
   /bes29/20260709_143629.jpg
   /bes29/20260709_143644.jpg

bes21
   /bes21/20260709_143442.jpg
   /bes21/20260709_143443.jpg


In [ ]:
pd.DataFrame(results)

In [ ]:
pd.DataFrame(results).to_csv('results.csv')

# EXP

In [ ]:
STOP

In [ ]:
# Process one image
file_in = np.random.choice(glob.glob(f'{folder_in_main}/*/*'))
print(file_in)

img = cv2.imread(file_in)
grid_spacing = find_grid_spacing(img, visualize = False)
predictor = prepare_model(verbose = False)
mask = make_mask(img, predictor, visualize = False)
ellipse = fit_ellipse(mask)

plt.figure()
plt.subplot(131)
plt.imshow(img)
plt.subplot(132)
plt.imshow(mask)
plt.subplot(133)
plot_mask_ellipse(mask, ellipse)
plt.show()

